# Data Aggregation & Grouping

## Learning goals

By the end of this lesson, you should be able to:

- filter observations using multiple conditions;
- translate a question into **groups → measure → aggregation**;
- use `groupby()` to summarize numerical variables;
- count observations using `size()` and distinguish it from `count()`;
- apply several statistics with `agg()`;
- group by more than one variable;
- filter aggregated results and use `reset_index()`.

**Guiding question:** for each group, what do we want to calculate?


### From combining data to answering questions

Combining tables gives us a dataset containing the information we need. **Aggregation** helps us turn those individual records into business insights.

For example, after combining monthly deliveries, we could calculate the total distance by month. Here we will use a **separate orders dataset** so this notebook can run on its own.


## 1. From individual rows to business questions

A DataFrame usually stores individual observations as rows.

But many business questions are not about one individual row.

They ask questions such as:

- What is the **average order value by city**?
- How many orders were processed by each warehouse?
- What are the **minimum, maximum and median delivery times by transport type**?
- Which warehouses processed more than a certain number of orders?

These questions require **aggregation**.

We will use an e-commerce fulfillment dataset throughout the lesson.

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "warehouse": ["Madrid", "Madrid", "Barcelona", "Barcelona", "Madrid",
                  "Valencia", "Valencia", "Barcelona", "Madrid", "Valencia"],
    "shipping": ["Standard", "Express", "Standard", "Express", "Express",
                 "Standard", "Express", "Standard", "Standard", "Express"],
    "order_value": [120, 85, 210, 150, 95, 180, 130, 75, 240, 160],
    "delivery_days": [4, 2, 5, 2, 1, 4, 2, 6, 3, 2],
    "returned": ["No", "No", "Yes", "No", "No", "Yes", "No", "No", "No", "Yes"]
})

display(orders)

## 2. Filtering rows before aggregation

Sometimes a question applies only to a subset of the data.

For example:

> Which **Express** orders had a value greater than €100?

Boolean conditions produce `True` / `False` values for each row.

In [ ]:
express_orders = orders.loc[
    (orders["shipping"] == "Express") &
    (orders["order_value"] > 100)
]

display(express_orders)

### Combining conditions

Use:

- `&` for **AND**;
- `|` for **OR**;
- `~` for **NOT**.

Put each condition inside parentheses.

```python
df.loc[
    (condition_1) &
    (condition_2)
]
```

Filtering answers:

> **Which rows do I want to keep?**

Aggregation answers a different question:

> **How do I summarize those rows?**

**Remember:** the parentheses around each comparison matter. The symbols `and` and `or` do not replace `&` and `|` when comparing Pandas Series.


### 💡 Quick Check 1 — Complete the code

Keep only orders that:

- were processed in Madrid;
- have an `order_value` greater than 100.

In [ ]:
madrid_orders = orders.loc[
    (orders["warehouse"] == ________) &
    (orders["order_value"] > ________)
]

display(madrid_orders)

## 3. Aggregation and grouping

An aggregation reduces multiple values to a summary value.

Common aggregation functions include:

- `mean()` → arithmetic mean;
- `sum()` → total;
- `min()` → minimum;
- `max()` → maximum;
- `median()` → median;
- `count()` → number of non-missing values.

For example, we can calculate the average value of **all** orders:

In [ ]:
average_order_value = orders["order_value"].mean()

print("Average order value:", average_order_value)

But often we do not want one statistic for the entire dataset.

We want the statistic **for each category**.

For example:

> What is the average order value **for each warehouse**?

That is where `groupby()` becomes useful.

The important part is to translate the business question into those three decisions before writing code.

In [ ]:
average_by_warehouse = (
    orders
    .groupby("warehouse")["order_value"]
    .mean()
)

display(average_by_warehouse)

Notice the order of the operation:

```python
df.groupby("grouping_column")["measure_column"].aggregation()
```

Read it almost as a sentence:

> Group the DataFrame by warehouse, select order value, and calculate the mean.

### 💡 Quick Check 2 — Complete the code

Calculate the **maximum delivery time** for each warehouse.

In [ ]:
max_delivery = (
    orders
    .________("warehouse")[________]
    .________()
)

display(max_delivery)

## 4. Counting observations in each group

Sometimes the business question is simply:

> How many observations belong to each group?

`groupby().size()` counts rows in each group.

In [ ]:
orders_per_warehouse = (
    orders
    .groupby("warehouse")
    .size()
)

display(orders_per_warehouse)

### `size()`, `value_counts()` and `nunique()`

These methods answer related but different questions.

#### `nunique()`

```python
orders["warehouse"].nunique()
```

answers:

> **How many different warehouses are there?**

It returns one number: the number of unique values.

#### `value_counts()`

```python
orders["warehouse"].value_counts()
```

answers:

> **How many times does each warehouse appear?**

For a single categorical column, this is often the simplest way to count frequencies.

#### `groupby().size()`

```python
orders.groupby("warehouse").size()
```

also answers:

> **How many rows belong to each warehouse?**

For one column, the result is conceptually very similar to `value_counts()`.

However, `groupby().size()` fits naturally into the general grouping workflow and becomes especially useful when grouping by several variables:

```python
orders.groupby(["warehouse", "shipping"]).size()
```

A useful distinction is:

| Question | Tool |
|---|---|
| How many different warehouses exist? | `nunique()` |
| How many rows belong to each warehouse? | `value_counts()` or `groupby().size()` |
| How many rows belong to each warehouse **and shipping method**? | `groupby([...]).size()` |

`count()` is slightly different again: it counts **non-missing values** in a selected column, while `size()` counts rows.

## 5. Multiple aggregations with `agg()`

Often one statistic is not enough.

Suppose we want to understand the distribution of order values by warehouse.

We can calculate several statistics at once with `agg()`.

The aggregation names in the list are strings. They tell Pandas which calculations to apply to each group.


In [ ]:
order_value_summary = (
    orders
    .groupby("warehouse")["order_value"]
    .agg(["count", "mean", "min", "max", "median"])
)

display(order_value_summary)

The result gives one row per warehouse and one column per aggregation.

This is useful when several statistics describe the **same measure**.

### 💡 Quick Check 4 — Complete the code

For each shipping method, calculate the **mean**, **minimum** and **maximum** delivery time.

In [ ]:
delivery_summary = (
    orders
    .________("shipping")[________]
    .________([________, ________, ________])
)

display(delivery_summary)

## 6. Different aggregations for different columns

Sometimes we want different calculations for different variables.

For example, for each warehouse we might want:

- the **sum** and **mean** of `order_value`;
- the **mean** and **maximum** of `delivery_days`.

A dictionary passed to `agg()` makes this explicit.

In [ ]:
warehouse_summary = (
    orders
    .groupby("warehouse")
    .agg({
        "order_value": ["sum", "mean"],
        "delivery_days": ["mean", "max"]
    })
)

display(warehouse_summary)

Think of the dictionary as instructions:

```python
{
    "column_1": ["aggregation_1", "aggregation_2"],
    "column_2": ["aggregation_3"]
}
```

This is one of the most flexible patterns for descriptive analysis with Pandas.

## 7. Grouping by multiple variables

A business question may define groups using more than one category.

For example:

> What is the average order value for each **warehouse and shipping method**?

Pass a list of columns to `groupby()`.

In [ ]:
warehouse_shipping = (
    orders
    .groupby(["warehouse", "shipping"])["order_value"]
    .mean()
)

display(warehouse_shipping)

The logic remains exactly the same:

```text
Groups?              → warehouse + shipping
Column to aggregate? → order_value
Aggregation?         → mean
```

## 8. Filtering aggregated results

There are two different moments when filtering may happen.

### Before aggregation

Filter **individual rows** first:

```text
rows → filter → group → aggregate
```

Example:

> Average order value by warehouse **only for returned orders**.

**Ask first:** are we excluding individual observations, or excluding groups after calculating their summaries?


In [ ]:
returned_average = (
    orders.loc[orders["returned"] == "Yes"]
    .groupby("warehouse")["order_value"]
    .mean()
)

display(returned_average)

### After aggregation

Sometimes we first calculate a result for every group and then keep only groups that meet a condition:

```text
rows → group → aggregate → filter result
```

Example:

> Which warehouses processed more than 3 orders?

In [ ]:
orders_per_warehouse = (
    orders
    .groupby("warehouse")
    .size()
)

busy_warehouses = orders_per_warehouse[
    orders_per_warehouse > 3
]

display(busy_warehouses)

These are different questions:

> **Filter before:** Which observations should participate in the calculation?

> **Filter after:** Which aggregated groups should appear in the result?

Keeping this distinction clear prevents many common mistakes.

## 9. Returning a DataFrame with `reset_index()`

A `groupby()` result often stores the grouping variable in the index.

If you want the grouping variable to become a normal column again, use `reset_index()`.

In [ ]:
warehouse_table = (
    orders
    .groupby("warehouse")["order_value"]
    .mean()
    .reset_index()
)

display(warehouse_table)

This is especially useful when the aggregated result will be:

- filtered further;
- merged with another DataFrame;
- exported;
- visualized later.

### A complete workflow: filter → group → aggregate → present

Imagine the operations manager asks:

> Among **Express** orders, what is the mean order value and the number of orders for each warehouse?

We filter **before** grouping because only Express orders should contribute to the calculation. We then use `agg()` and return a regular DataFrame.


In [ ]:
express_summary = (
    orders.loc[orders["shipping"] == "Express"]
    .groupby("warehouse")
    .agg(
        number_of_orders=("order_id", "size"),
        average_order_value=("order_value", "mean")
    )
    .reset_index()
)

display(express_summary)

The syntax `new_column_name=("original_column", "aggregation")` is called **named aggregation**. It lets us give descriptive names to the resulting columns. This is a useful alternative to the dictionary form of `agg()` shown above; you do not need to memorize both styles today.


## Optional final challenge — Support team performance (8–10 minutes)

If time allows, complete the following code using a new support-team dataset. This is a **consolidation activity**, not a new concept.

1. Average resolution time by team.
2. Minimum, maximum and median resolution time by team.
3. Average satisfaction by team and priority.
4. Count tickets per team and keep teams with more than two tickets.

If we are short on time, finish this independently after class.


In [ ]:
tickets = pd.DataFrame({
    "ticket_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "team": ["Billing", "Tech", "Tech", "Billing", "Sales", "Tech", "Sales", "Billing"],
    "priority": ["High", "Low", "High", "Low", "High", "Low", "Low", "High"],
    "resolution_hours": [5, 12, 8, 3, 6, 10, 4, 7],
    "satisfaction": [4, 3, 5, 4, 5, 4, 3, 5]
})

# 1. Average resolution time by team
avg_resolution = (
    tickets
    .________("team")[________]
    .________()
)

# 2. Min, max and median resolution time by team
resolution_summary = (
    tickets
    .________("team")[________]
    .________([________, ________, ________])
)

# 3. Average satisfaction by team and priority
satisfaction_summary = (
    tickets
    .________([________, ________])[________]
    .________()
)

# 4. Count tickets by team and keep teams with more than 2 tickets
ticket_counts = (
    tickets
    .________("team")
    .________()
)

busy_teams = ticket_counts[ticket_counts > ________]

display(avg_resolution)
display(resolution_summary)
display(satisfaction_summary)
display(busy_teams)

## Summary

Today you learned how to turn row-level data into group-level information.

The central workflow is:

**Filter → Group → Aggregate → Filter the result**

You practiced:

- boolean filtering with multiple conditions;
- `groupby()`;
- `size()` for counting rows;
- common aggregation functions;
- multiple aggregations with `agg()`;
- different aggregations for different columns;
- grouping by multiple variables;
- filtering aggregated results;
- `reset_index()`.

### Further exploration

Pandas provides additional GroupBy tools such as:

- iterating over a `GroupBy` object;
- the `.groups` attribute;
- `get_group()`;
- `transform()`;
- `filter()` on grouped data;
- named aggregations.

These tools can be introduced when an analysis requires them.